# Week 3 - Assignment: Voice Agent Development

From now on, we start to hands on buiding Research Voice Agent, truly useful AI Research Assistants must listen, understand, and respond with voice. **we will give you some simple introduction code as a starter, feel free to write your own code or do optimization.**

## 📚 Learning Objectives this week
to build a simple Voice Agent, we need these following knowledge.

* **1. Speech Recognition (ASR):** Convert audio to text using models like Whisper or Google Speech-to-Text.
* **2. Dialogue Generation with LLMs:** Feed transcribed user input into LLM (e.g. LLaMA 3) and generate natural language responses.
* **3. Text-to-Speech (TTS):** Use a TTS engine (CozyVoice) to convert generated responses into spoken audio.
* **4. FastAPI for API Serving:** Create a web server with FastAPI to handle audio file uploads and return voice responses.
* **5. Conversation State Management:** Track conversation history to enable multi-turn interaction.
* **6. Low-Latency Real-Time Processing:** Use asynchronous functions to reduce inference time and improve response experience.

---


> ✅ You do NOT need Docker. Just ensure your local Python environment works.

---

## 🧪 Project: Build an Local Voice Assistant

### 🎯 Goal:

Develop a real-time voice chatbot that can:

1. Take audio input via HTTP,
2. Transcribe audio to text (ASR),
3. Generate a response using LLM,
4. Convert the response back to speech (TTS),
5. Support 5-turn conversational memory.

---

### Step 1: FastAPI Skeleton

Create a simple FastAPI server that accepts an audio file via POST and returns an audio file in response:


here is the official guidance of FastAPI [fastapi](https://fastapi.tiangolo.com/)

In [12]:
pip install python-multipart

  Using cached python_multipart-0.0.20-py3-none-any.whl.metadata (1.8 kB)
Using cached python_multipart-0.0.20-py3-none-any.whl (24 kB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
pip install fastapi

Note: you may need to restart the kernel to use updated packages.


In [1]:
# !pip install FastAPI uvicorn 
# ! pip install python-multipart
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse

app = FastAPI()

@app.post("/chat/")
async def chat_endpoint(file: UploadFile = File(...)):
    audio_bytes = await file.read()
    # TODO: ASR → LLM → TTS
    return FileResponse("response.wav", media_type="audio/wav")


Run your server:

```bash
uvicorn main:app --reload
```

Test it with `curl`, Postman, or a custom frontend.

### Step 2: ASR (Speech Recognition)

Use OpenAI Whisper to transcribe the uploaded audio to text:

In [2]:
import whisper

asr_model = whisper.load_model("small")

def transcribe_audio(audio_bytes):
    with open("temp.wav", "wb") as f:
        f.write(audio_bytes)
    result = asr_model.transcribe("temp.wav")
    return result["text"]

ModuleNotFoundError: No module named 'whisper'

Add it to the `/chat/` route:

In [ ]:
user_text = transcribe_audio(audio_bytes)

Print `user_text` for debugging.


### Step 3: Response Generation (LLM)

Generate context-aware responses using Llama 3. Use HuggingFace `pipeline` to call LLaMA 3 or similar models:


In [ ]:
from transformers import pipeline
#llm = pipeline("text-generation", model="meta-llama/Llama-3-8B")
llm = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct")

conversation_history = []

def generate_response(user_text):
    conversation_history.append({"role": "user", "text": user_text})
    # Construct prompt from history
    prompt = ""
    for turn in conversation_history[-5:]:
        prompt += f"{turn['role']}: {turn['text']}\n"
    outputs = llm(prompt, max_new_tokens=100)
    bot_response = outputs[0]["generated_text"]
    conversation_history.append({"role": "assistant", "text": bot_response})
    return bot_response


Device set to use cpu


Call in route:

In [ ]:
user_text="Hello, how are you?"
bot_text = generate_response(user_text)
print(bot_text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


user: Hello, how are you?
I: I am fine, thank you.
user: How was your day?
I: It was okay, I'm just getting ready for work. I have a meeting at 10am.
I: I'm glad you're getting ready. Do you have any plans for the weekend?
I: No, I'm actually planning on going to the beach on Saturday.
user: That sounds great! I've been meaning to get there, I've heard it's beautiful.
I: Yeah, it



---



### Step 4: TTS (Text to Speech)


In [ ]:
pip install pyttsx3

  Using cached pywin32-311-cp312-cp312-win_amd64.whl.metadata (10 kB)
Using cached pywin32-311-cp312-cp312-win_amd64.whl (9.5 MB)

   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]
   ---------------------------------------- 0/4 [pywin32]

In [ ]:
import pyttsx3

engine = pyttsx3.init()
engine.setProperty("rate", 150)  # Speech rate
engine.setProperty("volume", 0.8)  # Volume 0-1

text = "Hello, this is a test of text to speech."
engine.say(text)
engine.runAndWait()
#engine.save_to_file(text, "output.wav")
#engine.runAndWait()

Convert LLM text responses to natural-sounding speech. \
try to use cozyvoice to complete Text to Speech, here is the original project.
[Cosyvoice](https://github.com/FunAudioLLM/CosyVoice)

In [ ]:
import cosyvoice
import pkgutil

for p in pkgutil.walk_packages(cosyvoice.__path__, cosyvoice.__name__ + "."):
    print(p.name)


In [ ]:
import sys
sys.path.append('third_party/Matcha-TTS')
from cosyvoice.cli.cosyvoice import CosyVoice, CosyVoice2
from cosyvoice.utils.file_utils import load_wav
import torchaudio

cosyvoice = CosyVoice2('pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, load_vllm=False, fp16=False)

# NOTE if you want to reproduce the results on https://funaudiollm.github.io/cosyvoice2, please add text_frontend=False during inference
# zero_shot usage
prompt_speech_16k = load_wav('./asset/zero_shot_prompt.wav', 16000)
for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '希望你以后能够做的比我还好呦。', prompt_speech_16k, stream=False)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)


ModuleNotFoundError: No module named 'cosyvoice.cli'

### Use it in the route:

In [ ]:
pythonaudio_path = synthesize_speech(bot_text)



---


### Step 5: Full Integration

Your final `/chat/` endpoint might look like this:

In [ ]:

@app.post("/chat/")
async def chat_endpoint(file: UploadFile = File(...)):
    audio_bytes = await file.read()
    user_text = transcribe_audio(audio_bytes)
    bot_text = generate_response(user_text)
    audio_path = synthesize_speech(bot_text)
    return FileResponse(audio_path, media_type="audio/wav")



---

## ✅ Deliverables

* [ ] A runnable FastAPI project with `/chat/` endpoint
* [ ] A working voice assistant that handles **5-turn** multi-round conversations
* [ ] Code with clear structure and modular components (ASR, LLM, TTS)
* [ ] A **2-minute screen recording** demo: record 5 turns of real-time interaction
* [ ] Optional: Add conversation memory display, prompt formatting logic, async optimization

---

## 🌟 Extension Ideas (Optional)

* Use `async` processing for parallel ASR/LLM/TTS.
* Integrate a microphone frontend UI for live recording.
* Add speaker identification or personalized voice response.

---
